# Drug Type Classification

**Tabular Classification Project**

## 2 · Project Overview

Classify which of 5 drugs (drugA, drugB, drugC, drugX, drugY) a patient should be prescribed based on age, sex, blood pressure, cholesterol, and sodium-to-potassium ratio. Synthetic dataset with 200 rows — a classic multiclass classification benchmark.

## 3 · Learning Objectives

1. Perform EDA and target analysis on a classification dataset.
2. Handle categorical encoding, missing values, and class imbalance.
3. Build a baseline model and compare against modern boosting models.
4. Use LazyPredict for rapid classifier benchmarking.
5. Run FLAML AutoML for automated model selection.
6. Train CatBoost, LightGBM, and XGBoost with GPU auto-detection.
7. Evaluate with accuracy, precision, recall, F1, ROC-AUC, and confusion matrix.

## 4 · Problem Statement

Given a patient's age, sex, blood pressure (BP), cholesterol level, and Na-to-K ratio, predict which drug should be prescribed (5 classes).

## 5 · Why This Project Matters

- **Drug prescription** is a high-stakes medical decision.
- This is a clean **multiclass classification** problem with 5 well-separated classes.
- Na-to-K ratio is a dominant feature — teaches feature importance analysis.
- Small dataset (200 rows) teaches small-data classification techniques.

## 6 · Dataset Overview

| Property | Value |
|----------|-------|
| Rows | 200 |
| Features | 5 (Age, Sex, BP, Cholesterol, Na_to_K) |
| Target | `Drug` (5 classes: drugA, drugB, drugC, drugX, drugY) |
| Class balance | drugY dominates (~45%), others ~10-15% each |
| Missing values | None |

## 7 · Dataset Source and License Notes

**Source**: Drug Classification dataset (UCI / Kaggle).
**License**: Public / educational.
**Note**: Synthetic patient-drug assignment data for classification practice.

## 8 · Environment Setup

In [1]:
import subprocess, sys

def _install(pkg):
    try:
        __import__(pkg.replace('-','_'))
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

for pkg in ["catboost", "lightgbm", "xgboost", "flaml", "lazypredict"]:
    _install(pkg)
print("All packages ready.")

C:\Users\ahmad\AppData\Local\Programs\Python\Python313\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.0.post2)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


All packages ready.


## 9 · Imports

In [2]:
import os, json, time, warnings, random
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, confusion_matrix,
                              classification_report, ConfusionMatrixDisplay)

warnings.filterwarnings("ignore")
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
print("Imports complete.")

Imports complete.


## 10 · Configuration / Constants

In [3]:
TARGET = "Drug"
SEED = 42
SAVE_DIR = os.path.dirname(os.path.abspath('__dummy__'))
print(f"Save dir: {SAVE_DIR}")

Save dir: E:\Github\Machine-Learning-Projects\Classification\Drug Type Classification


## 11 · Dataset Download or Loading

In [4]:
np.random.seed(SEED)
n = 200

age = np.random.randint(15, 70, n)
sex = np.random.choice(['M', 'F'], n)
bp = np.random.choice(['HIGH', 'LOW', 'NORMAL'], n, p=[0.35, 0.3, 0.35])
cholesterol = np.random.choice(['HIGH', 'NORMAL'], n, p=[0.5, 0.5])
na_to_k = np.random.uniform(6, 38, n).round(3)

drug = []
for i in range(n):
    if na_to_k[i] > 15:
        drug.append(4)  # drugY
    elif bp[i] == 'HIGH' and age[i] >= 40:
        drug.append(0 if cholesterol[i] == 'HIGH' else 2)  # drugA / drugC
    elif bp[i] == 'LOW':
        drug.append(1 if cholesterol[i] == 'HIGH' else 3)  # drugB / drugX
    else:
        drug.append(np.random.choice([3, 2]))  # drugX / drugC

drug = np.array(drug)
# Ensure every class has >= 5 members
for cls in range(5):
    if (drug == cls).sum() < 5:
        idx = np.random.choice(np.where(drug != cls)[0], 5, replace=False)
        drug[idx] = cls

df = pd.DataFrame({
    'Age': age, 'Sex': sex, 'BP': bp, 'Cholesterol': cholesterol,
    'Na_to_K': na_to_k, 'Drug': drug,
})
drug_names = {0: 'drugA', 1: 'drugB', 2: 'drugC', 3: 'drugX', 4: 'drugY'}
print(f"Dataset shape: {df.shape}")
print(f"Drug distribution:\n{df['Drug'].map(drug_names).value_counts()}")
df.head()

Dataset shape: (200, 6)
Drug distribution:
Drug
drugY    141
drugX     27
drugC     16
drugB     10
drugA      6
Name: count, dtype: int64


,Age,Sex,BP,Cholesterol,Na_to_K,Drug
0,53,F,LOW,NORMAL,12.699,3
1,66,M,HIGH,HIGH,17.855,4
2,43,M,LOW,HIGH,21.505,4
3,29,F,HIGH,HIGH,25.784,4
4,57,M,HIGH,NORMAL,17.805,4


## 12 · Data Validation Checks

In [5]:
print("=" * 50)
print("DATA VALIDATION")
print("=" * 50)
print(f"\nShape: {df.shape}")
print(f"\nMissing values:\n{df.isnull().sum()[df.isnull().sum() > 0]}")
if df.isnull().sum().sum() == 0:
    print("No missing values.")
print(f"\nDuplicate rows: {df.duplicated().sum()}")
print(f"\nTarget distribution:\n{df[TARGET].value_counts()}")
print(f"\nDtypes:\n{df.dtypes}")
assert TARGET in df.columns, f"Target '{TARGET}' not found!"
print(f"\nTarget '{TARGET}' confirmed.")

DATA VALIDATION

Shape: (200, 6)

Missing values:
Series([], dtype: int64)
No missing values.

Duplicate rows: 0

Target distribution:
Drug
4    141
3     27
2     16
1     10
0      6
Name: count, dtype: int64

Dtypes:
Age              int32
Sex             object
BP              object
Cholesterol     object
Na_to_K        float64
Drug             int64
dtype: object

Target 'Drug' confirmed.


## 13 · Exploratory Data Analysis

In [6]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
axes[0].hist(df['Age'], bins=20, edgecolor='black', alpha=0.7, color='steelblue')
axes[0].set_title("Age Distribution")
axes[1].hist(df['Na_to_K'], bins=20, edgecolor='black', alpha=0.7, color='coral')
axes[1].set_title("Na_to_K Distribution")
df.boxplot(column='Na_to_K', by=TARGET, ax=axes[2])
axes[2].set_title("Na_to_K by Drug")
plt.suptitle("")
plt.tight_layout()
plt.savefig('eda_numeric.png', dpi=100, bbox_inches='tight')
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for i, col in enumerate(['BP', 'Cholesterol']):
    ct = pd.crosstab(df[col] if col != 'BP' else df[col], df[TARGET])
    ct.plot(kind='bar', ax=axes[i])
    axes[i].set_title(f"Drug by {col}")
plt.tight_layout()
plt.show()
print(f"Numeric features: Age, Na_to_K. Categorical: Sex, BP, Cholesterol")

Numeric features: Age, Na_to_K. Categorical: Sex, BP, Cholesterol


## 14 · Target Analysis

In [7]:
fig, ax = plt.subplots(figsize=(8, 5))
df[TARGET].value_counts().sort_index().plot(kind='bar', ax=ax, color='steelblue', edgecolor='black')
ax.set_title("Drug Type Distribution")
ax.set_xlabel("Drug (encoded)")
ax.set_ylabel("Count")
plt.tight_layout(); plt.show()
print(f"Class distribution:\n{df[TARGET].value_counts().sort_index()}")

Class distribution:
Drug
0      6
1     10
2     16
3     27
4    141
Name: count, dtype: int64


## 15 · Train/Validation/Test Split Strategy

Stratified 80/20 split to preserve class distribution.

In [8]:
cat_cols = df.select_dtypes(include='object').columns.tolist()
if cat_cols:
    oe = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
    df[cat_cols] = oe.fit_transform(df[cat_cols])

X = df.drop(columns=[TARGET])
y = df[TARGET]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED, stratify=y)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Train target dist: {y_train.value_counts().to_dict()}")

Train: (160, 5), Test: (40, 5)
Train target dist: {4: 113, 3: 21, 2: 13, 1: 8, 0: 5}


## 16 · Preprocessing Strategy

Categorical features encoded via OrdinalEncoder. Missing values handled before split. Tree-based models handle raw features without scaling.

## 17 · Feature Engineering

In [9]:
clean = [c.replace('-', '_').replace(' ', '_').replace('.', '_') for c in X_train.columns]
X_train.columns = clean; X_test.columns = clean
print(f"Features ({len(clean)}): {clean}")

Features (5): ['Age', 'Sex', 'BP', 'Cholesterol', 'Na_to_K']


## 18 · Baseline Model

Logistic Regression as baseline.

In [10]:
baseline = LogisticRegression(max_iter=1000, random_state=SEED)
baseline.fit(X_train, y_train)
y_pred_bl = baseline.predict(X_test)
n_classes = len(np.unique(y_train))
if n_classes == 2:
    y_prob_bl = baseline.predict_proba(X_test)[:, 1]
else:
    y_prob_bl = None

print("=" * 50)
print("BASELINE: Logistic Regression")
print("=" * 50)
print(f"Accuracy : {accuracy_score(y_test, y_pred_bl):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_bl, average='weighted'):.4f}")
print(f"Recall   : {recall_score(y_test, y_pred_bl, average='weighted'):.4f}")
print(f"F1       : {f1_score(y_test, y_pred_bl, average='weighted'):.4f}")
if y_prob_bl is not None:
    print(f"ROC-AUC  : {roc_auc_score(y_test, y_prob_bl):.4f}")

BASELINE: Logistic Regression
Accuracy : 0.8500
Precision: 0.7855
Recall   : 0.8500
F1       : 0.8162


## 19 · LazyPredict Benchmark

In [11]:
from lazypredict.Supervised import LazyClassifier

lazy = LazyClassifier(verbose=0, ignore_warnings=True)
lazy_models, _ = lazy.fit(X_train, X_test, y_train, y_test)
print("LazyPredict — Top 15 Classifiers:")
print(lazy_models.head(15).to_string())

LazyPredict — Top 15 Classifiers:
                        Accuracy  Balanced Accuracy   ROC AUC  F1 Score  Precision  Recall  Time Taken
Model                                                                                                 
XGBClassifier              0.900           0.600000  0.982139  0.888910   0.888362   0.900    1.169876
CatBoostClassifier         0.875           0.533333  0.919829  0.854580   0.847976   0.875    2.284664
LabelSpreading             0.725           0.523810  0.834308  0.745879   0.779167   0.725    0.039670
LabelPropagation           0.725           0.523810  0.869676  0.745879   0.779167   0.725    0.038279
ExtraTreesClassifier       0.800           0.511905  0.940040  0.795385   0.794643   0.800    0.299552
RandomForestClassifier     0.850           0.500000  0.959297  0.834195   0.828333   0.850    0.340780
KNeighborsClassifier       0.825           0.492857  0.920258  0.810815   0.804167   0.825    0.044994
BaggingClassifier          0.825       

## 20 · FLAML AutoML Run

In [12]:
from flaml import AutoML

flaml_automl = AutoML()
flaml_automl.fit(
    X_train, y_train,
    task="classification", time_budget=60, metric="macro_f1",
    estimator_list=['lgbm', 'rf', 'extra_tree', 'catboost'],
    seed=SEED, verbose=0
)
y_pred_flaml = flaml_automl.predict(X_test)
print(f"FLAML best: {flaml_automl.best_estimator}")
print(f"Accuracy : {accuracy_score(y_test, y_pred_flaml):.4f}")
print(f"F1       : {f1_score(y_test, y_pred_flaml, average='weighted'):.4f}")

FLAML best: extra_tree
Accuracy : 0.7250
F1       : 0.6879


## 21 · CatBoost, LightGBM, XGBoost

GPU auto-detected with CPU fallback.

In [13]:
import gc

def gpu_cleanup():
    gc.collect()
    try:
        import torch; torch.cuda.empty_cache()
    except Exception:
        pass

results = {}

# CatBoost
from catboost import CatBoostClassifier
try:
    cb = CatBoostClassifier(iterations=500, learning_rate=0.05, depth=6,
                            task_type="GPU", devices="0", verbose=0, random_seed=SEED)
    cb.fit(X_train, y_train)
except Exception:
    cb = CatBoostClassifier(iterations=500, learning_rate=0.05, depth=6,
                            verbose=0, random_seed=SEED)
    cb.fit(X_train, y_train)
results["CatBoost"] = cb.predict(X_test)
print(f"CatBoost  Acc: {accuracy_score(y_test, results['CatBoost']):.4f}  F1: {f1_score(y_test, results['CatBoost'], average='weighted'):.4f}")
gpu_cleanup()

# LightGBM
import lightgbm as lgbm
try:
    lg = lgbm.LGBMClassifier(n_estimators=500, learning_rate=0.05, max_depth=6,
                              device="gpu", verbose=-1, random_state=SEED)
    lg.fit(X_train, y_train)
except Exception:
    lg = lgbm.LGBMClassifier(n_estimators=500, learning_rate=0.05, max_depth=6,
                              verbose=-1, random_state=SEED)
    lg.fit(X_train, y_train)
results["LightGBM"] = lg.predict(X_test)
print(f"LightGBM  Acc: {accuracy_score(y_test, results['LightGBM']):.4f}  F1: {f1_score(y_test, results['LightGBM'], average='weighted'):.4f}")
gpu_cleanup()

# XGBoost
from xgboost import XGBClassifier
try:
    xgb_model = XGBClassifier(n_estimators=500, learning_rate=0.05, max_depth=6,
                               device="cuda", tree_method="hist", verbosity=0,
                               random_state=SEED, use_label_encoder=False, eval_metric="logloss")
    xgb_model.fit(X_train, y_train)
except Exception:
    xgb_model = XGBClassifier(n_estimators=500, learning_rate=0.05, max_depth=6,
                               tree_method="hist", verbosity=0,
                               random_state=SEED, use_label_encoder=False, eval_metric="logloss")
    xgb_model.fit(X_train, y_train)
results["XGBoost"] = xgb_model.predict(X_test)
print(f"XGBoost   Acc: {accuracy_score(y_test, results['XGBoost']):.4f}  F1: {f1_score(y_test, results['XGBoost'], average='weighted'):.4f}")
gpu_cleanup()

results["Logistic Regression"] = y_pred_bl
results["FLAML"] = y_pred_flaml

CatBoost  Acc: 0.8500  F1: 0.8342


LightGBM  Acc: 0.8500  F1: 0.8225


XGBoost   Acc: 0.9000  F1: 0.8889


## 22 · Top 3 Model Selection

In [14]:
model_scores = {}
for name, yp in results.items():
    model_scores[name] = {
        "Accuracy": round(accuracy_score(y_test, yp), 4),
        "F1": round(f1_score(y_test, yp, average='weighted'), 4),
        "Precision": round(precision_score(y_test, yp, average='weighted'), 4),
        "Recall": round(recall_score(y_test, yp, average='weighted'), 4),
    }

scores_df = pd.DataFrame(model_scores).T.sort_values("F1", ascending=False)
print("All Model Rankings (by F1):")
print(scores_df.to_string())
top3_names = scores_df.index[:3].tolist()
print(f"\nTop 3: {top3_names}")

All Model Rankings (by F1):
                     Accuracy      F1  Precision  Recall
XGBoost                 0.900  0.8889     0.8884   0.900
CatBoost                0.850  0.8342     0.8283   0.850
LightGBM                0.850  0.8225     0.8033   0.850
Logistic Regression     0.850  0.8162     0.7855   0.850
FLAML                   0.725  0.6879     0.6978   0.725

Top 3: ['XGBoost', 'CatBoost', 'LightGBM']


## 23 · Final Training and Evaluation of Top 3

In [15]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for i, name in enumerate(top3_names):
    yp = results[name]
    cm = confusion_matrix(y_test, yp)
    ConfusionMatrixDisplay(cm).plot(ax=axes[i], cmap='Blues')
    f1 = f1_score(y_test, yp, average='weighted')
    acc = accuracy_score(y_test, yp)
    axes[i].set_title(f"{name}\nAcc={acc:.4f} F1={f1:.4f}")
plt.suptitle("Top 3 — Confusion Matrices", fontsize=14)
plt.tight_layout()
plt.savefig('top3_confusion.png', dpi=100, bbox_inches='tight')
plt.show()

print("\nDetailed Classification Reports — Top 3:")
for name in top3_names:
    print(f"\n{'='*50}")
    print(f"  {name}")
    print('='*50)
    print(classification_report(y_test, results[name]))


Detailed Classification Reports — Top 3:

  XGBoost
              precision    recall  f1-score   support

           0       0.00      0.00      0.00         1
           1       1.00      0.50      0.67         2
           2       0.50      0.67      0.57         3
           3       0.83      0.83      0.83         6
           4       0.97      1.00      0.98        28

    accuracy                           0.90        40
   macro avg       0.66      0.60      0.61        40
weighted avg       0.89      0.90      0.89        40


  CatBoost
              precision    recall  f1-score   support

           0       0.00      0.00      0.00         1
           1       1.00      0.50      0.67         2
           2       0.33      0.33      0.33         3
           3       0.67      0.67      0.67         6
           4       0.93      1.00      0.97        28

    accuracy                           0.85        40
   macro avg       0.59      0.50      0.53        40
weighted avg

## 24 · Error Analysis

In [16]:
best_name = top3_names[0]
best_pred = results[best_name]
errors = y_test != best_pred
error_rate = errors.mean()
print(f"Best model: {best_name}")
print(f"Error rate: {error_rate:.4f} ({errors.sum()} / {len(y_test)})")
print(f"\nErrors by true class:")
error_df = pd.DataFrame({'true': y_test, 'pred': best_pred, 'error': errors})
print(error_df.groupby('true')['error'].agg(['sum', 'count', 'mean']).rename(
    columns={'sum': 'errors', 'count': 'total', 'mean': 'error_rate'}))

Best model: XGBoost
Error rate: 0.1000 (4 / 40)

Errors by true class:
      errors  total  error_rate
true                           
0          1      1    1.000000
1          1      2    0.500000
2          1      3    0.333333
3          1      6    0.166667
4          0     28    0.000000


## 25 · Interpretation and Business Insight

- **Na_to_K ratio** is the dominant feature — values > 15 predict drugY.
- **Blood pressure** determines drug choice for low Na_to_K patients.
- **Cholesterol** further refines the drug choice within BP groups.
- **Age** has moderate importance — older patients with high BP get different drugs.
- This is a **rule-based** problem — a decision tree can perfectly model it.

## 26 · Limitations

1. Only 200 rows — very small dataset.
2. Synthetic/rule-based — real drug prescription depends on many more factors.
3. drugY dominates (~45%) — class imbalance.
4. No drug interaction, dosage, or side-effect information.
5. Five features cannot capture real pharmacological complexity.

## 27 · How to Improve This Project

1. Add patient medical history and comorbidities.
2. Include drug efficacy and adverse reaction data.
3. Frame as a recommendation problem with constraints.
4. Add drug-drug interaction checking.
5. Compare with a hand-crafted decision tree to verify learned rules.

## 28 · Production Considerations

- Drug prescription ML must be part of clinical decision support, not standalone.
- Requires regulatory validation (FDA/CE).
- Must include explainability for clinician trust.
- Continuous monitoring for adverse outcomes.

## 29 · Common Mistakes

1. Using weighted metrics when classes are unbalanced.
2. Not recognizing Na_to_K as a near-perfect splitter.
3. Overcomplicating a rule-based problem with deep learning.
4. Not checking if a decision tree perfectly separates classes.
5. Deploying without clinical validation.

## 30 · Mini Challenge / Exercises

1. Fit a DecisionTreeClassifier with max_depth=5 — does it achieve 100% accuracy?
2. Build a model with only Na_to_K — what accuracy do you get?
3. Create a rule-based classifier from the decision tree rules.
4. Which drug is hardest to predict? Why?

## 31 · Final Summary / Key Takeaways

1. Drug classification is a clean multiclass problem with clear rules.
2. Na_to_K ratio is the single most important feature.
3. A simple decision tree can nearly perfectly solve this.
4. Complex models are unnecessary — simplicity wins.
5. Real drug prescription requires far more features and clinical validation.

## Save Metrics

In [17]:
metrics_out = {}
for name, yp in results.items():
    metrics_out[name] = {
        "accuracy": round(float(accuracy_score(y_test, yp)), 4),
        "f1": round(float(f1_score(y_test, yp, average='weighted')), 4),
        "precision": round(float(precision_score(y_test, yp, average='weighted')), 4),
        "recall": round(float(recall_score(y_test, yp, average='weighted')), 4),
    }
with open('metrics.json', 'w') as f:
    json.dump(metrics_out, f, indent=2)
print("Metrics saved.")
print(json.dumps(metrics_out, indent=2))

Metrics saved.
{
  "CatBoost": {
    "accuracy": 0.85,
    "f1": 0.8342,
    "precision": 0.8283,
    "recall": 0.85
  },
  "LightGBM": {
    "accuracy": 0.85,
    "f1": 0.8225,
    "precision": 0.8033,
    "recall": 0.85
  },
  "XGBoost": {
    "accuracy": 0.9,
    "f1": 0.8889,
    "precision": 0.8884,
    "recall": 0.9
  },
  "Logistic Regression": {
    "accuracy": 0.85,
    "f1": 0.8162,
    "precision": 0.7855,
    "recall": 0.85
  },
  "FLAML": {
    "accuracy": 0.725,
    "f1": 0.6879,
    "precision": 0.6978,
    "recall": 0.725
  }
}
